In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display options
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [2]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-11-29 23:27:08.834782-05:00


#### Build X/y, make reproducible folds

In [5]:
import pandas as pd
import numpy as np
from sqlalchemy import text
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer

# 1) Load provider train/test
df_train = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
df_test = pd.read_sql("SELECT * FROM mart.provider_test",  con=engine)

# 2) Identify id + label + features
id_col = [c for c in df_train.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
label_col = "label_provider_fraud_1_0"

# Drop obvious non-feature cols (ids, dates, the label)
drop_cols = {id_col, label_col}
feature_cols = [c for c in df_train.columns if c not in drop_cols]

# 3) Minimal cleaning
X_full = df_train[feature_cols]
y_full = df_train[label_col].astype(int)

imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(
    X_full), columns=feature_cols, index=df_train.index)

# 4) Train/valid holdout (stratified)
X_tr, X_val, y_tr, y_val, prov_tr, prov_val = train_test_split(
    X_imp, y_full, df_train[id_col],
    test_size=0.2, stratify=y_full, random_state=42
)

# 5) 5-fold CV assignments
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = np.empty(len(df_train), dtype=int)
folds[:] = -1
for k, (_, val_idx) in enumerate(skf.split(X_imp, y_full)):
    folds[val_idx] = k
cv_df = pd.DataFrame({id_col: df_train[id_col], "cv_fold": folds})

# 6) Class imbalance summary & suggested scale_pos_weight (for XGBoost/LightGBM)
pos = int((y_full == 1).sum())
neg = int((y_full == 0).sum())
ratio = neg / max(1, pos)
print({"n_providers": len(y_full), "positives": pos,
      "negatives": neg, "neg_to_pos_ratio": ratio})
# For XGBoost: scale_pos_weight ≈ neg/pos; for LightGBM: `is_unbalance=True` or same ratio param.

# 7) Persist CV folds back to DB for reproducibility
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS mart.provider_cv_folds"))
    conn.execute(text(f"""
        CREATE TABLE mart.provider_cv_folds AS
        SELECT t.{id_col}::{'text' if df_train[id_col].dtype == 'object' else 'bigint'} AS {id_col},
               f.cv_fold
        FROM mart.provider_train t
        JOIN (VALUES {','.join([f"('{pid}',{fold})" if isinstance(pid, str) else f"({pid},{fold})"
                                for pid, fold in zip(cv_df[id_col], cv_df['cv_fold'])])}) AS f({id_col},cv_fold)
        ON t.{id_col} = f.{id_col};
    """))

{'n_providers': 5410, 'positives': 506, 'negatives': 4904, 'neg_to_pos_ratio': 9.691699604743082}
